# MODUL PRAKTIKUM BIG DATA
## Pertemuan 4 — Instalasi Apache Spark & Pengenalan PySpark DataFrame

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 4 |
| **Topik** | Konsep Apache Spark, Instalasi Spark + PySpark, Operasi Dasar DataFrame |
| **Estimasi Waktu** | 3 x 50 menit |
| **Prasyarat** | Modul Pertemuan 1-3 selesai (VM Ubuntu, environment `bigdata`, Hadoop + HDFS aktif) |

---

> **Konsistensi versi:** Modul ini menggunakan **Apache Spark 3.5.9** (rilis LTS terbaru pada versi 3.5, per pertengahan 2026) dan **PySpark 3.5.9** — versi paket Python-nya **wajib sama persis** dengan versi Spark standalone yang terpasang, jika tidak akan muncul error yang membingungkan. Spark 3.5.x resmi mendukung **Python 3.8 hingga 3.11** — persis mencakup Python 3.11 pada environment `bigdata` kita, sehingga **tidak perlu membuat environment baru**. Spark 3.5.x juga kompatibel penuh dengan **Java 11** dan **Hadoop 3.4.3** yang sudah anda pasang di Pertemuan 3. Seluruh kombinasi ini sudah diverifikasi saling cocok — jangan mengganti versi secara sembarangan.

> **Mengapa Spark, padahal sudah ada Hadoop MapReduce?** MapReduce klasik (Pertemuan 3) menulis data sementara ke disk di setiap tahap pemrosesan — lambat untuk beban kerja iteratif. **Spark memproses data di dalam memori (RAM)**, membuatnya bisa 10-100x lebih cepat untuk banyak kasus, sambil tetap dapat membaca/menulis data dari **HDFS** yang sama.


---
## Recap Pertemuan Sebelumnya

Sebelum memulai, pastikan:
- [ ] VM Ubuntu menyala, `conda activate bigdata` berhasil
- [ ] Hadoop aktif: jalankan `start-dfs.sh` dan `start-yarn.sh`, verifikasi dengan `jps` (harus tampil 5 proses seperti Pertemuan 3)
- [ ] Folder `~/praktikum-bigdata` masih berisi seluruh modul & dataset sebelumnya

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 4, mahasiswa mampu:
1. Menjelaskan konsep dasar Apache Spark dan perbedaannya dengan MapReduce klasik.
2. Memasang dan mengonfigurasi Apache Spark beserta PySpark.
3. Membuat `SparkSession` dan memuat data ke dalam Spark DataFrame dari disk lokal maupun HDFS.
4. Melakukan operasi dasar DataFrame: `select`, `filter`, `groupBy`, `agg`.
5. Membandingkan sintaks PySpark DataFrame dengan pandas DataFrame yang telah dipelajari di Pertemuan 2.

---

## 4.1 Konsep Dasar Apache Spark

| Komponen Arsitektur | Peran |
|---|---|
| **Driver Program** | Proses utama yang menjalankan kode Spark kita (mis. script Python), menyusun rencana eksekusi |
| **Cluster Manager** | Mengalokasikan sumber daya — bisa berupa mode *local*, *Standalone*, atau **YARN** (yang sudah kita install di Pertemuan 3!) |
| **Executor** | Proses pekerja yang benar-benar menjalankan tugas komputasi dan menyimpan data di memori |

**RDD vs DataFrame:**
- **RDD** (*Resilient Distributed Dataset*) adalah struktur data dasar & paling awal di Spark — kumpulan objek terdistribusi tanpa struktur kolom yang jelas.
- **DataFrame** dibangun di atas RDD, terstruktur seperti tabel (mirip pandas DataFrame atau tabel SQL), **jauh lebih mudah digunakan dan lebih dioptimalkan secara otomatis**. Praktikum ini akan berfokus pada DataFrame karena inilah yang paling banyak dipakai di dunia kerja saat ini.

> **Mode yang kita gunakan:** Pada praktikum ini, Spark dijalankan dalam **mode local** (`local[*]`) — seluruh proses (driver & executor) berjalan di satu VM yang sama, memanfaatkan seluruh core CPU yang tersedia. Ini adalah cara standar untuk *belajar* sintaks PySpark. Spark yang sama juga bisa dijalankan di atas **YARN** (memanfaatkan cluster Hadoop sungguhan) tanpa mengubah kode Python-nya sama sekali — hanya berbeda di konfigurasi saat menjalankan.


---
## 4.2 Download dan Ekstrak Apache Spark 3.5.9

**Jalankan di Terminal:**

```bash
cd ~
wget https://downloads.apache.org/spark/spark-3.5.9/spark-3.5.9-bin-hadoop3.tgz
```

> Kita download varian **`bin-hadoop3`** — dibangun untuk bekerja dengan Hadoop versi 3.x seperti yang sudah kita install. Jika link tidak aktif, kunjungi **https://spark.apache.org/downloads.html**, pilih versi **3.5.9**, paket type **"Pre-built for Apache Hadoop 3.3 and later"**, lalu salin link nya.

Ekstrak dan pindahkan:

```bash
tar -xzvf spark-3.5.9-bin-hadoop3.tgz
mv spark-3.5.9-bin-hadoop3 spark
```

Verifikasi:

```bash
ls ~/spark
```

Anda seharusnya melihat folder seperti `bin`, `sbin`, `python`, `jars`, dan lainnya — struktur serupa dengan folder `hadoop` di Pertemuan 3.

---

## 4.3 Konfigurasi Environment Variables

Edit kembali `~/.bashrc`:

```bash
nano ~/.bashrc
```

Tambahkan baris berikut **di bagian paling bawah** (di bawah konfigurasi Hadoop yang sudah ada dari Pertemuan 3):

```bash
# Konfigurasi Spark — Praktikum Big Data
export SPARK_HOME=$HOME/spark
export PATH=$PATH:$SPARK_HOME/bin:$SPARK_HOME/sbin
export PYSPARK_PYTHON=python3
```

Simpan (`Ctrl+O`, Enter) dan keluar (`Ctrl+X`), lalu muat ulang:

```bash
source ~/.bashrc
```

Verifikasi:

```bash
echo $SPARK_HOME
spark-submit --version
```

Output seharusnya menampilkan versi **Spark 3.5.9** beserta logo ASCII Spark.

---

## 4.4 Instalasi PySpark di Environment `bigdata`

Meski Spark standalone sudah terpasang, kita tetap perlu install package **`pyspark`** di dalam conda environment `bigdata` agar dapat dipanggil dari Python/Jupyter. **Versi wajib sama persis** dengan Spark standalone di atas (3.5.9) untuk menghindari error yang membingungkan.

**Jalankan di Terminal (pastikan `(bigdata)` aktif):**

```bash
conda activate bigdata
pip install pyspark==3.5.9
```

Verifikasi instalasi:

```bash
python -c "import pyspark; print('PySpark', pyspark.__version__)"
```

Output yang diharapkan:

```
PySpark 3.5.9
```


---
## Melanjutkan di Jupyter Notebook

Buka Terminal baru, jalankan:

```bash
conda activate bigdata
cd ~/praktikum-bigdata
jupyter notebook
```

Buka modul Pertemuan 4 ini di Jupyter, lalu lanjutkan dari sini menggunakan code cell.

## 4.5 Membuat SparkSession

`SparkSession` adalah **titik masuk utama** untuk bekerja dengan Spark DataFrame — mirip seperti membuka koneksi sebelum bisa menggunakan Spark, wajib dibuat di awal setiap script/notebook PySpark.

In [2]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

26/09/12 17:04:51 WARN Utils: Your hostname, irkham resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/12 17:04:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/12 17:04:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


Jika muncul `Versi Spark: 3.5.9` tanpa error, instalasi anda berhasil sempurna.

---

## 4.6 Memuat Data ke Spark DataFrame

Kita akan menggunakan kembali dataset transaksi e-commerce dari Pertemuan 2 sebagai bahan latihan (jika berkas `data_transaksi_ecommerce.csv` sudah tidak ada di folder ini, jalankan dulu cell pembuatan datanya dari modul Pertemuan 2, atau gunakan cell di bawah untuk membuatnya ulang).

In [3]:
# Membuat ulang dataset contoh (identik dengan Tugas Mandiri Pertemuan 2) jika belum ada
import os
if not os.path.exists("data_transaksi_ecommerce.csv"):
    import numpy as np
    import pandas as pd
    np.random.seed(42)
    n = 600
    kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
    kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
    metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
    tanggal_range = pd.date_range("2026-07-01", "2026-07-31", freq="D")
    data = {
        "order_id": [f"ORD-{1000+i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25,0.25,0.20,0.15,0.15]),
        "kota": np.random.choice(kota_list, size=n),
        "unit_terjual": np.random.randint(1, 10, size=n),
        "harga_satuan": np.random.choice([25000,50000,75000,100000,150000,250000,500000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n, p=[0.35,0.30,0.20,0.15]),
    }
    pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False)
    print("Dataset dibuat ulang.")
else:
    print("Dataset sudah tersedia.")

Dataset sudah tersedia.


In [3]:
# Membaca berkas CSV lokal menjadi Spark DataFrame
# header=True    -> baris pertama dianggap nama kolom
# inferSchema=True -> Spark otomatis menebak tipe data tiap kolom (angka, teks, dst.)
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)

print("Tipe objek:", type(df))
df.printSchema()

[Stage 1:>                                                          (0 + 1) / 1]

Tipe objek: <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)



Perhatikan `printSchema()` menampilkan struktur kolom beserta tipe datanya — sangat berguna untuk memverifikasi Spark membaca data dengan benar, terutama pada dataset besar yang tidak mungkin dilihat seluruhnya sekaligus.

In [4]:
# Menampilkan beberapa baris pertama — mirip df.head() di pandas, namun disebut show()
df.show(5)

# Menghitung jumlah baris — mirip len(df) di pandas
print("Jumlah baris:", df.count())

+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-07 00:00:00|Kesehatan & Kecan...|  Magelang|           9|      250000|         E-Wallet|
|ORD-1001|2026-07-20 00:00:00|   Makanan & Minuman|Yogyakarta|           9|       50000|     Kartu Kredit|
|ORD-1002|2026-07-29 00:00:00|             Fashion| Purworejo|           2|       50000|     Kartu Kredit|
|ORD-1003|2026-07-15 00:00:00|          Elektronik|      Solo|           3|       50000|         E-Wallet|
|ORD-1004|2026-07-11 00:00:00|          Elektronik|  Semarang|           5|      250000|    Transfer Bank|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
only showing top 5 rows



[Stage 3:>                                                          (0 + 1) / 1]

Jumlah baris: 600


### Bridging dari pandas ke PySpark

Anda sudah mengenal operasi-operasi berikut di pandas (Pertemuan 2). Berikut ini adalah kesamaan nya di PySpark:

| Operasi | pandas | PySpark |
|---|---|---|
| Lihat data teratas | `df.head()` | `df.show(5)` |
| Jumlah baris | `len(df)` | `df.count()` |
| Info struktur | `df.info()` | `df.printSchema()` |
| Pilih kolom | `df[["kolom1","kolom2"]]` | `df.select("kolom1","kolom2")` |
| Filter baris | `df[df["kolom"] > 5]` | `df.filter(df["kolom"] > 5)` |
| Kelompokkan data | `df.groupby("kolom").sum()` | `df.groupBy("kolom").sum()` |

**Perbedaan paling mendasar:** pandas memuat *seluruh* data ke RAM dan mengeksekusi setiap baris kode **langsung**. PySpark bersifat ***lazy evaluation*** — kode seperti `filter()` atau `select()` **tidak langsung dieksekusi**, melainkan baru disusun rencananya, dan baru benar-benar dijalankan ketika kita memanggil *action* seperti `.show()`, `.count()`, atau `.collect()`. Hal ini memungkinkan Spark mengoptimalkan keseluruhan rencana eksekusi sebelum benar-benar memprosesnya — sangat penting ketika data berukuran sangat besar.


---
## 4.7 Operasi Dasar DataFrame: Select, Filter, GroupBy, Agg

### 4.7.1 `select()` — Memilih Kolom Tertentu

In [5]:
df.select("order_id", "kota", "kategori").show(5)

+--------+----------+--------------------+
|order_id|      kota|            kategori|
+--------+----------+--------------------+
|ORD-1000|  Magelang|Kesehatan & Kecan...|
|ORD-1001|Yogyakarta|   Makanan & Minuman|
|ORD-1002| Purworejo|             Fashion|
|ORD-1003|      Solo|          Elektronik|
|ORD-1004|  Semarang|          Elektronik|
+--------+----------+--------------------+
only showing top 5 rows



### 4.7.2 `filter()` — Menyaring Baris

In [6]:
from pyspark.sql.functions import col

# Menyaring transaksi kategori Elektronik dengan unit_terjual lebih dari 5
df.filter((col("kategori") == "Elektronik") & (col("unit_terjual") > 5)).show(5)

+--------+-------------------+----------+----------+------------+------------+-----------------+
|order_id|            tanggal|  kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+----------+----------+------------+------------+-----------------+
|ORD-1019|2026-07-03 00:00:00|Elektronik|  Magelang|           9|       75000|         E-Wallet|
|ORD-1020|2026-07-22 00:00:00|Elektronik|  Semarang|           6|      150000|     Kartu Kredit|
|ORD-1028|2026-07-28 00:00:00|Elektronik|  Semarang|           8|      250000|    Transfer Bank|
|ORD-1040|2026-07-31 00:00:00|Elektronik| Purworejo|           6|      100000|         E-Wallet|
|ORD-1054|2026-07-25 00:00:00|Elektronik|Yogyakarta|           9|      500000|    Transfer Bank|
+--------+-------------------+----------+----------+------------+------------+-----------------+
only showing top 5 rows



> Fungsi `col("nama_kolom")` digunakan untuk merujuk sebuah kolom secara eksplisit — praktik standar di PySpark, terutama saat menuliskan kondisi kompleks seperti di atas.

### 4.7.3 `groupBy()` dan `agg()` — Meringkas Data

In [7]:
from pyspark.sql.functions import sum as spark_sum, count, avg

# Menambahkan kolom baru: total_pendapatan = unit_terjual x harga_satuan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# Meringkas: total pendapatan & jumlah transaksi per kota, diurutkan dari tertinggi
ringkasan_kota = df.groupBy("kota").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan"),
    count("order_id").alias("jumlah_transaksi"),
    avg("unit_terjual").alias("rata_rata_unit")
).orderBy(col("total_pendapatan").desc())

ringkasan_kota.show()

+----------+----------------+----------------+-----------------+
|      kota|total_pendapatan|jumlah_transaksi|   rata_rata_unit|
+----------+----------------+----------------+-----------------+
|  Magelang|       109925000|             119| 5.07563025210084|
| Purworejo|       104675000|             122|4.737704918032787|
|Yogyakarta|       104625000|             129|5.248062015503876|
|  Semarang|        87200000|             118|4.771186440677966|
|      Solo|        83950000|             112|5.053571428571429|
+----------+----------------+----------------+-----------------+



> **Perhatikan:** kita menggunakan `sum as spark_sum` saat impor — ini untuk menghindari bentrok dengan fungsi `sum()` bawaan Python. Kebiasaan baik yang perlu selalu diingat saat bekerja dengan PySpark.

---

## 4.8 Bonus: Membaca Data Langsung dari HDFS

Karena Hadoop/HDFS sudah aktif sejak Pertemuan 3, Spark dapat membaca data **langsung dari HDFS** tanpa perlu download ke disk lokal terlebih dahulu — inilah kekuatan sesungguhnya dari kombinasi Spark + Hadoop.

In [8]:
# Upload dataset ke HDFS terlebih dahulu (jika belum ada dari Pertemuan 3)
!hdfs dfs -mkdir -p /user/irkham/pertemuan4
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/irkham/pertemuan4/

# Membaca CSV LANGSUNG dari HDFS menggunakan Spark — perhatikan prefix "hdfs://"
df_dari_hdfs = spark.read.csv(
    "hdfs://localhost:9000/user/irkham/pertemuan4/data_transaksi_ecommerce.csv",
    header=True, inferSchema=True
)
print("Jumlah baris dari HDFS:", df_dari_hdfs.count())
df_dari_hdfs.show(5)

Jumlah baris dari HDFS: 600
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-07 00:00:00|Kesehatan & Kecan...|  Magelang|           9|      250000|         E-Wallet|
|ORD-1001|2026-07-20 00:00:00|   Makanan & Minuman|Yogyakarta|           9|       50000|     Kartu Kredit|
|ORD-1002|2026-07-29 00:00:00|             Fashion| Purworejo|           2|       50000|     Kartu Kredit|
|ORD-1003|2026-07-15 00:00:00|          Elektronik|      Solo|           3|       50000|         E-Wallet|
|ORD-1004|2026-07-11 00:00:00|          Elektronik|  Semarang|           5|      250000|    Transfer Bank|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
only show

Jika cell di atas berhasil menampilkan data, anda baru saja membuktikan bahwa **HDFS (Pertemuan 3) dan Spark (Pertemuan 4) sudah terintegrasi sepenuhnya** — inilah fondasi arsitektur Big Data sesungguhnya: HDFS sebagai lapisan penyimpanan, Spark sebagai lapisan pemrosesan.

---

## Menutup SparkSession

Kebiasaan baik: selalu tutup SparkSession di akhir notebook untuk clear up resources (memori & CPU) yang dipakainya.

In [1]:
spark.stop()
print("SparkSession ditutup.")

NameError: name 'spark' is not defined

> Jika ingin melanjutkan latihan setelah ini, jalankan ulang cell pembuatan `SparkSession` pada Sub-bab 4.5.


---
## Latihan Mandiri

Buat SparkSession baru terlebih dahulu (copy cell dari Sub-bab 4.5) sebelum mengerjakan latihan berikut.

In [4]:
# Jalankan ini dulu sebelum mengerjakan latihan di bawah
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
print("Siap. Jumlah baris:", df.count())

[Stage 2:>                                                          (0 + 1) / 1]

Siap. Jumlah baris: 600


**Soal 1.** Tampilkan hanya kolom `order_id`, `kategori`, dan `total_pendapatan` untuk transaksi dengan `metode_pembayaran` bernilai `"E-Wallet"`.

In [6]:
# Jawaban Soal 1 di sini
df.filter(col("metode_pembayaran") == "E-Wallet").select("order_id", "kategori", "total_pendapatan").show()

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1000|Kesehatan & Kecan...|         2250000|
|ORD-1003|          Elektronik|          150000|
|ORD-1006|   Makanan & Minuman|           25000|
|ORD-1013|   Makanan & Minuman|          900000|
|ORD-1014|             Fashion|           75000|
|ORD-1015|          Elektronik|          375000|
|ORD-1017|Kesehatan & Kecan...|         4000000|
|ORD-1019|          Elektronik|          675000|
|ORD-1021|          Elektronik|          200000|
|ORD-1022|        Rumah Tangga|          900000|
|ORD-1023|Kesehatan & Kecan...|         2500000|
|ORD-1024|             Fashion|          750000|
|ORD-1027|             Fashion|           50000|
|ORD-1029|             Fashion|          600000|
|ORD-1030|             Fashion|          300000|
|ORD-1033|          Elektronik|          300000|
|ORD-1035|   Makanan & Minuman|         2500000|
|ORD-1036|   Makanan

**Soal 2.** Hitung total pendapatan **per kategori** (bukan per kota), urutkan dari yang tertinggi.

In [8]:
# Jawaban Soal 2 di sini
from pyspark.sql.functions import sum as spark_sum, count, avg

df.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(col("total_pendapatan").desc()).show()

[Stage 16:>                                                         (0 + 1) / 1]

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|             Fashion|       124825000|
|          Elektronik|       110600000|
|   Makanan & Minuman|        95400000|
|Kesehatan & Kecan...|        82200000|
|        Rumah Tangga|        77350000|
+--------------------+----------------+



**Soal 3.** Tampilkan jumlah transaksi untuk masing-masing `metode_pembayaran` (tidak pakai `agg`).

In [5]:
# Jawaban Soal 3 di sini
df.groupBy("metode_pembayaran").count().show()

[Stage 5:>                                                          (0 + 1) / 1]

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD|  114|
|    Transfer Bank|  208|
|     Kartu Kredit|   85|
|         E-Wallet|  193|
+-----------------+-----+



**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: apa yang dimaksud dengan *lazy evaluation* di PySpark, dan mengapa hal ini menguntungkan ketika bekerja dengan data berskala besar? Tulis jawaban pada markdown cell di bawah ini.

*(Tulis jawaban di sini)*.
Lazy Evaluation artinya operasi operasi seperti filter,select,groypBy tidak langsung dijalankan saat ditulis.Proses akan dijalankan ketika memanggil action seperti show,count,collect.Mengguntungkan karena dapat mengoptimalkan seluruh eksekusi sebelum memproses sehingga lebih hemat waktu,memori,efisien dengan data skala besar

---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

>  **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 5 dimulai**.
>  **Sifat tugas:** individu.

### Konteks / Skenario

Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas **dipindahkan ke PySpark**, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, **membaca data langsung dari HDFS**.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset baru (transaksi bulan September 2026, lebih banyak baris dari sebelumnya) dan mengunggahnya ke HDFS.

In [7]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/irkham/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/irkham/tugas4/
print("Berhasil diunggah ke HDFS: /user/irkham/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/irkham/tugas4/transaksi_september_2026.csv


### Instruksi Pengerjaan

Buat notebook baru **`Tugas4_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` baru, lalu kerjakan bagian **A sampai E** berikut — **seluruhnya wajib menggunakan PySpark, bukan pandas**, dan data **wajib dibaca langsung dari HDFS** (`hdfs://localhost:9000/...`), bukan dari berkas lokal.

---

**A. Membaca dan Eksplorasi Awal** *(bobot 15%)*

Baca dataset dari HDFS, tampilkan `printSchema()`, jumlah baris (`count()`), dan 10 baris pertama (`show(10)`).

**B. Menangani Data Kosong** *(bobot 15%)*

Kolom `rating` memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan `df.na.fill()` atau `df.na.drop()` (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

**C. Transformasi Data** *(bobot 20%)*

Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`), lalu tambahkan kolom `tier_transaksi` yang bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya 

**D. Analisis dengan GroupBy** *(bobot 30%)*

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki `total_pendapatan` tertinggi?
2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?
3. Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran` (data kosong sudah ditangani di bagian B)?

**E. Menyimpan Hasil ke HDFS** *(bobot 20%)*

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

> **Catatan:** Spark menyimpan hasil sebagai **beberapa berkas partisi** (`part-00000...`, dst.), bukan satu berkas tunggal seperti pandas — ini normal dan justru mencerminkan sifat terdistribusi Spark. Jelaskan secara singkat pada markdown cell mengapa hal ini terjadi

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas4_[NIM]_[Nama Lengkap].ipynb` melalui ELITA, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.
- **Dilarang menggunakan pandas** untuk bagian analisis A-E (boleh dipakai hanya di sel pembuatan dataset yang sudah disediakan).

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Baca & Eksplorasi | Data berhasil dibaca dari HDFS, struktur & jumlah baris benar | 15% |
| B. Data Kosong | Missing value teridentifikasi & ditangani dengan alasan yang logis | 15% |
| C. Transformasi | Kedua kolom baru dihitung dengan benar menggunakan fungsi PySpark yang tepat | 20% |
| D. Analisis GroupBy | Ketiga jawaban analitis benar & menggunakan sintaks PySpark (bukan pandas) | 30% |
| E. Simpan ke HDFS | Hasil berhasil tersimpan ke HDFS; penjelasan tentang partisi tepat | 20% |
